In [5]:
%pip install pandas

^C
Note: you may need to restart the kernel to use updated packages.


     ---------------------------------------- 10.0/10.0 MB 6.0 MB/s eta 0:00:00
     ---------------------------------------- 12.6/12.6 MB 6.4 MB/s eta 0:00:00
     -------------------------------------- 348.2/348.2 kB 7.3 MB/s eta 0:00:00



[notice] A new release of pip available: 22.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import pandas as pd

# Load dataset
df = pd.read_csv("new_transactions.csv")

# Create Flourish network graph dataset
graph_df = df[
    ["from_account", "to_account", "is_laundering"]
].copy()

# Rename columns to Flourish format
graph_df.rename(
    columns={
        "from_account": "source",
        "to_account": "target"
    },
    inplace=True
)

# Convert laundering flag:
# False -> 0
# True  -> 500
graph_df["value"] = graph_df["is_laundering"].astype(bool).astype(int) * 500

# Keep only required columns
graph_df = graph_df[
    ["source", "target", "value"]
]

# Clean account IDs
graph_df["source"] = graph_df["source"].astype(str).str.strip()
graph_df["target"] = graph_df["target"].astype(str).str.strip()

# Remove invalid rows
graph_df = graph_df[
    graph_df["source"].notna() &
    graph_df["target"].notna() &
    (graph_df["source"] != "") &
    (graph_df["target"] != "")
]

# Save
graph_df.to_csv("graph.csv", index=False)

# Summary
print("Graph dataset created successfully!")
print("----------------------------------")
print(f"Total edges:        {len(graph_df):,}")
print(f"Unique source nodes: {graph_df['source'].nunique():,}")
print(f"Unique target nodes: {graph_df['target'].nunique():,}")
print(
    f"Unique nodes overall: "
    f"{pd.unique(pd.concat([graph_df['source'], graph_df['target']])).size:,}"
)
print(f"Fraud edges (500):   {(graph_df['value'] == 500).sum():,}")
print(f"Normal edges (0):    {(graph_df['value'] == 0).sum():,}")

print("\nColumns:")
print(graph_df.columns.tolist())

print("\nFirst 10 rows:")
display(graph_df.head(10))

Graph dataset created successfully!
----------------------------------
Total edges:        6,924,049
Unique source nodes: 681,281
Unique target nodes: 576,176
Unique nodes overall: 705,903
Fraud edges (500):   3,565
Normal edges (0):    6,920,484

Columns:
['source', 'target', 'value']

First 10 rows:


,source,target,value
0,8000ECA90,8000ECA90,0
1,80021DAD0,80021DAD0,0
2,8000ECA90,8006AA910,0
3,8006AD080,8006AD080,0
4,8006AD530,8006AD530,0
5,8006ADD30,8006ADD30,0
6,800059120,8006AD4E0,0
7,8000ECA90,8000ECA90,0
8,8006AA910,81470DCF0,0
9,8006AD4E0,8006AD4E0,0


In [ ]:
import pandas as pd

# Load transactions
df = pd.read_csv("new_transactions.csv")

# Keep only fraud/laundering transactions
fraud_graph = df[
    df["is_laundering"] == True
][[
    "from_account",
    "to_account"
]].copy()

# Rename for Flourish
fraud_graph.rename(
    columns={
        "from_account": "source",
        "to_account": "target"
    },
    inplace=True
)

# Give every fraud edge a value of 500
fraud_graph["value"] = 500

# Remove self-loops
fraud_graph = fraud_graph[
    fraud_graph["source"] != fraud_graph["target"]
]

# Remove duplicate edges if you want a cleaner network
# Comment this out if you want every individual transaction represented.
fraud_graph = fraud_graph.drop_duplicates(
    subset=["source", "target"]
)

# Save
fraud_graph.to_csv(
    "fraud_graph.csv",
    index=False
)

# Summary
print("Fraud graph created!")
print("--------------------")
print(f"Fraud transactions: {len(fraud_graph):,}")
print(f"Unique source accounts: {fraud_graph['source'].nunique():,}")
print(f"Unique destination accounts: {fraud_graph['target'].nunique():,}")
print(
    f"Unique accounts overall: "
    f"{pd.unique(pd.concat([fraud_graph['source'], fraud_graph['target']])).size:,}"
)

print("\nColumns:")
print(fraud_graph.columns.tolist())

display(fraud_graph.head(20))

In [7]:
import pandas as pd

df = pd.read_csv("new_transactions.csv")

src = df[["from_account", "is_laundering"]].rename(
    columns={"from_account": "account"}
)

dst = df[["to_account", "is_laundering"]].rename(
    columns={"to_account": "account"}
)

activity = pd.concat([src, dst])

summary = activity.groupby("account")["is_laundering"].agg(
    total="count",
    laundering="sum"
).reset_index()

summary["normal"] = (
    summary["total"] - summary["laundering"]
)

# Use pandas conditions instead of np.select
summary["type"] = "UNKNOWN"

summary.loc[
    summary["laundering"] == 0,
    "type"
] = "NORMAL_ONLY"

summary.loc[
    summary["normal"] == 0,
    "type"
] = "LAUNDERING_ONLY"

summary.loc[
    (summary["laundering"] > 0) &
    (summary["normal"] > 0),
    "type"
] = "MIXED"

print(summary["type"].value_counts())
      
print("\nPercentage:")
print(
    summary["type"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

type
NORMAL_ONLY        700599
MIXED                5258
LAUNDERING_ONLY        46
Name: count, dtype: int64

Percentage:
type
NORMAL_ONLY        99.25
MIXED               0.74
LAUNDERING_ONLY     0.01
Name: proportion, dtype: float64


In [8]:
laundering = df[df["is_laundering"] == True].copy()

incoming = laundering.groupby("to_account").size()
outgoing = laundering.groupby("from_account").size()

flow = pd.concat(
    [incoming.rename("incoming"),
     outgoing.rename("outgoing")],
    axis=1
).fillna(0)

endpoints = flow[
    (flow["incoming"] > 0) &
    (flow["outgoing"] == 0)
].sort_values("incoming", ascending=False)

print("Laundering transactions:", len(laundering))
print("Potential endpoints:", len(endpoints))

print("\nTop endpoints:")
print(endpoints.head(30))

Laundering transactions: 3565
Potential endpoints: 2922

Top endpoints:
           incoming  outgoing
800A08E20      15.0       0.0
8012FC120      15.0       0.0
80021A8B0      14.0       0.0
800AF0650      13.0       0.0
800299A70      12.0       0.0
800648CD0      10.0       0.0
801127C90      10.0       0.0
80151E540      10.0       0.0
800130E70      10.0       0.0
8016B3750       8.0       0.0
807F00750       6.0       0.0
800914620       6.0       0.0
8022EC780       6.0       0.0
800318E70       4.0       0.0
807445340       4.0       0.0
800775100       3.0       0.0
800C4B600       3.0       0.0
80D7D0A70       3.0       0.0
8000A94C0       3.0       0.0
808DCBA80       2.0       0.0
800DECA20       2.0       0.0
8021D6600       2.0       0.0
809927D20       2.0       0.0
8087D2320       2.0       0.0
80031CB90       2.0       0.0
80151B8A0       2.0       0.0
80460B920       2.0       0.0
8001AA5A0       2.0       0.0
800853210       2.0       0.0
80033E730       2.0       0.

In [9]:
from collections import defaultdict
import pandas as pd

# Make sure timestamp is datetime
laundering = df[
    df["is_laundering"] == True
].copy()

laundering["timestamp"] = pd.to_datetime(
    laundering["timestamp"],
    errors="coerce"
)

# Remove rows with invalid timestamps
laundering = laundering.dropna(
    subset=["timestamp"]
)

laundering = laundering.sort_values("timestamp")

# Build outgoing transaction lookup
outgoing = defaultdict(list)

for _, r in laundering.iterrows():
    outgoing[r["from_account"]].append(r)


def trace_chain(start, max_steps=10):

    chain = [start]
    current = start
    last_time = pd.Timestamp.min

    for _ in range(max_steps):

        candidates = [
            x for x in outgoing[current]
            if x["timestamp"] > last_time
        ]

        if not candidates:
            break

        # Earliest valid next laundering transaction
        tx = min(
            candidates,
            key=lambda x: x["timestamp"]
        )

        next_account = tx["to_account"]

        chain.append(next_account)

        current = next_account
        last_time = tx["timestamp"]

    return chain


# Accounts that appear as laundering senders
# but never receive laundering money
sources = (
    set(laundering["from_account"])
    - set(laundering["to_account"])
)

print("Potential laundering sources:", len(sources))
print("\nExample laundering chains:\n")

shown = 0

for source in sources:

    chain = trace_chain(source)

    if len(chain) >= 2:

        print(" → ".join(map(str, chain)))

        shown += 1

        if shown >= 50:
            break

Potential laundering sources: 2000

Example laundering chains:

817F61290 → 817F61560
81537B0A0 → 81554E720
819B96580 → 819B96670
8010325A0 → 801032A00
80FD16F10 → 80FD17910
8026B40E0 → 8026B48A0
807542250 → 80A547970
818F20580 → 818F21230
81BFD1230 → 81BFD3D90
8000F66E0 → 8006B57C0
805571950 → 80557F410
800241280 → 800471960
808704180 → 806A48300
8071111A0 → 8105AC9A0
80AF45B10 → 80AF46D50
80810AD90 → 803279B20
81A6181C0 → 81A618260
80B395700 → 80B3AC8C0
807BA5D00 → 808CF6590
810738CC0 → 81073E500
8131C4210 → 8131F52A0
8192F2D20 → 8192F3480
800FAB8A0 → 80CAFB7F0 → 80BE9B140
81A76FC00 → 81A76FC50
8008D1FA0 → 80025E130 → 8018CA480
80C2FFF10 → 80C300800
819B54E60 → 819B54F00
802EBBE10 → 802EBBF50
805756EB0 → 80F98ECB0
81B80C371 → 81B80CDF1
818E6A2C0 → 818E7CBB0
803C17870 → 803C17C20
8003E4420 → 800ED4BA0
8036B18C0 → 8036D8030
800107BB0 → 800139150
818827730 → 818827870
8196F4481 → 8196F71A1
810D31310 → 810D31360
80F5B3260 → 80F5B38F0
8047ECB60 → 8047ED1F0
804C4A0C0 → 804C64EC0
80CB8B410 

In [10]:
endpoint_ids = set(endpoints.index)

endpoint_data = df[
    df["to_account"].isin(endpoint_ids)
].copy()

endpoint_data = endpoint_data.sort_values("timestamp")

endpoint_data = endpoint_data[[
    "timestamp",
    "from_account",
    "to_account",
    "amount_paid",
    "payment_mode",
    "from_district",
    "to_district",
    "is_laundering"
]]

print(endpoint_data.head(50))

endpoint_data.to_csv(
    "laundering_endpoint_transactions.csv",
    index=False
)

                  timestamp from_account to_account  amount_paid  \
205     2022-09-01 00:00:00    80005CA70  80005CA70     20021.25   
25458   2022-09-01 00:00:00    803823440  803823440     26261.54   
410881  2022-09-01 00:00:00    816213A00  816213A00     97563.88   
208565  2022-09-01 00:00:00    808A4BA90  808A4BA90       235.15   
200357  2022-09-01 00:00:00    806CF5D20  806CF5D20        19.53   
37084   2022-09-01 00:00:00    805D9F700  805D9FF90     27103.67   
38271   2022-09-01 00:00:00    10042B660  805F7F2B0      4968.53   
410993  2022-09-01 00:00:00    81688F900  81688F900       510.02   
192009  2022-09-01 00:00:00    8040C4E10  8023FA820         1.61   
188889  2022-09-01 00:00:00    803154300  803154300         5.47   
385923  2022-09-01 00:00:00    8128A5080  8128A5080       129.50   
57607   2022-09-01 00:00:00    800BCD3F0  80899AFF0     30674.76   
183477  2022-09-01 00:00:00    802847600  802847600        11.83   
383078  2022-09-01 00:00:00    812342540  812342

In [11]:
import pandas as pd

df = endpoint_data

# Keep ONLY laundering/fraud transactions
fraud = df[
    df["is_laundering"] == True
].copy()

fraud["timestamp"] = pd.to_datetime(
    fraud["timestamp"],
    errors="coerce"
)

print("Total transactions:", len(df))
print("Laundering transactions:", len(fraud))

print("\nPayment mode distribution:")
payment_dist = (
    fraud["payment_mode"]
    .value_counts()
    .to_frame("transaction_count")
)

payment_dist["percentage"] = (
    payment_dist["transaction_count"]
    / len(fraud) * 100
).round(2)

print(payment_dist)

# Save isolated fraud dataset
fraud.to_csv(
    "fraud_only_transactions.csv",
    index=False
)

# Extract temporal features
fraud["date"] = fraud["timestamp"].dt.date
fraud["hour"] = fraud["timestamp"].dt.hour
fraud["day_of_week"] = fraud["timestamp"].dt.day_name()
fraud["month"] = fraud["timestamp"].dt.to_period("M").astype(str)

print("=== BY HOUR ===")
hour_dist = (
    fraud["hour"]
    .value_counts()
    .sort_index()
)

print(hour_dist)

print("\n=== BY DAY OF WEEK ===")
dow_dist = (
    fraud["day_of_week"]
    .value_counts()
)

print(dow_dist)

print("\n=== BY MONTH ===")
month_dist = (
    fraud["month"]
    .value_counts()
    .sort_index()
)

print(month_dist)

# Payment mode vs hour
mode_hour = pd.crosstab(
    fraud["hour"],
    fraud["payment_mode"]
)

print("=== PAYMENT MODE × HOUR ===")
print(mode_hour)

print("\n=== PAYMENT MODE × HOUR (%) ===")

mode_hour_pct = (
    pd.crosstab(
        fraud["hour"],
        fraud["payment_mode"],
        normalize="columns"
    ) * 100
).round(2)

print(mode_hour_pct)

Total transactions: 53233
Laundering transactions: 3099

Payment mode distribution:
                transaction_count  percentage
payment_mode                                 
IMPS                         2158       69.64
UPI                           713       23.01
ATM Withdrawal                122        3.94
Other Digital                 106        3.42
=== BY HOUR ===
hour
0     122
1      98
2     123
3     130
4     114
5     110
6     103
7     126
8     126
9     127
10    144
11    146
12    148
13    150
14    148
15    151
16    154
17    118
18    137
19    139
20    137
21    127
22    114
23    107
Name: count, dtype: int64

=== BY DAY OF WEEK ===
day_of_week
Friday       664
Thursday     645
Saturday     500
Monday       344
Tuesday      341
Wednesday    338
Sunday       267
Name: count, dtype: int64

=== BY MONTH ===
month
2022-09    3099
Name: count, dtype: int64
=== PAYMENT MODE × HOUR ===
payment_mode  ATM Withdrawal  IMPS  Other Digital  UPI
hour                   

In [12]:
import pandas as pd

df = pd.read_csv("new_transactions.csv")

# Clean fields
df["amount_paid"] = pd.to_numeric(df["amount_paid"], errors="coerce").fillna(0)

df["is_laundering"] = (
    df["is_laundering"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("true")
)

# -----------------------------
# INBOUND transactions
# -----------------------------
inbound = (
    df.groupby("to_account")
    .agg(
        total_inbound_amount=("amount_paid", "sum"),
        inbound_transactions=("amount_paid", "count")
    )
)

# -----------------------------
# OUTBOUND transactions
# -----------------------------
outbound = (
    df.groupby("from_account")
    .agg(
        outbound_transactions=("amount_paid", "count"),
        total_outbound_amount=("amount_paid", "sum")
    )
)

# -----------------------------
# FRAUDULENT inbound transactions
# -----------------------------
fraud_inbound = (
    df[df["is_laundering"] == True]
    .groupby("to_account")
    .agg(
        fraud_received_balance=("amount_paid", "sum"),
        fraud_received_transactions=("amount_paid", "count")
    )
)

# -----------------------------
# Combine everything
# -----------------------------
accounts = inbound.join(outbound, how="left")
accounts = accounts.join(fraud_inbound, how="left")

accounts = accounts.fillna(0)

# -----------------------------
# Required accounts:
#   1. Have inbound transactions
#   2. Have ZERO outbound transactions
#   3. Have at least one fraudulent inbound transaction
# -----------------------------
accounts = accounts[
    (accounts["inbound_transactions"] > 0) &
    (accounts["outbound_transactions"] == 0) &
    (accounts["fraud_received_transactions"] > 0)
].copy()

# -----------------------------
# Calculate useful ratios
# -----------------------------
accounts["fraud_transaction_percentage"] = (
    accounts["fraud_received_transactions"]
    / accounts["inbound_transactions"]
    * 100
).round(2)

accounts["fraud_amount_percentage"] = (
    accounts["fraud_received_balance"]
    / accounts["total_inbound_amount"]
    * 100
).round(2)

# Since there is no actual balance field,
# use total received amount as the account balance proxy
accounts["total_account_balance"] = (
    accounts["total_inbound_amount"]
)

# -----------------------------
# Final columns
# -----------------------------
accounts = accounts.reset_index()

accounts = accounts.rename(
    columns={"to_account": "account"}
)

accounts = accounts[
    [
        "account",
        "total_account_balance",
        "fraud_received_balance",
        "total_inbound_amount",
        "inbound_transactions",
        "fraud_received_transactions",
        "fraud_transaction_percentage",
        "fraud_amount_percentage"
    ]
]

# Highest fraud-received accounts first
accounts = accounts.sort_values(
    "fraud_received_balance",
    ascending=False
)

# Save CSV
accounts.to_csv(
    "fraud_endpoint_accounts.csv",
    index=False
)

print("CSV created: fraud_endpoint_accounts.csv")
print("Accounts found:", len(accounts))
print("\nTop 20:")
print(accounts.head(20).to_string(index=False))

CSV created: fraud_endpoint_accounts.csv
Accounts found: 165

Top 20:
  account  total_account_balance  fraud_received_balance  total_inbound_amount  inbound_transactions  fraud_received_transactions  fraud_transaction_percentage  fraud_amount_percentage
806A6A3C0           7.240056e+09             61442793.05          7.240056e+09                    34                          1.0                          2.94                     0.85
80A09F7C0           1.955515e+08              2083392.14          1.955515e+08                    27                          1.0                          3.70                     1.07
81072A9F0           1.122109e+07               975390.85          1.122109e+07                    27                          1.0                          3.70                     8.69
80A717F50           1.653174e+07               508340.78          1.653174e+07                    17                          1.0                          5.88                     3.07
8083C